# SOC L2 Lab — Forensic Log Analysis & Telemetry Correlation

This notebook is designed for **SOC L2 Analysts** to investigate, analyze, and visualize telemetry generated across the cyber range by `Invoke-SocLabLogGeneration.ps1`.

### Analyzed Log Source Coverage:
* **Windows Endpoint & Active Directory (P02, P03, P14, P15):** Events `4625` (Brute force), `4688` (Suspicious process creation), and `4720`–`4738` (AD account/group manipulation).
* **Linux Server & Applications (P04, P04B, P11, P13):** `Syslog` (`sshd`, `sudo`, `su`), Apache logs, and PostgreSQL authentication.
* **Web Services & PaaS (P05, P12):** `AppServiceHTTPLogs` (web scanning) and `W3CIISLog` (Windows IIS).
* **Cloud Control Plane & Kubernetes (P07, P08, P09, P10):** `AZKVAuditLogs` (Key Vault), `AKSAuditAdmin` (K8s API), `ContainerLogV2`, and `AzureActivity`.

---

In [ ]:
# Install required Python packages in Azure ML
%pip install -q azure-identity azure-monitor-query pandas matplotlib


## 1. Initialization & Connection to Microsoft Sentinel / Log Analytics

Uses the Compute Instance (`soc-l2-ci`) system-assigned Managed Identity to authenticate via `DefaultAzureCredential` against workspace `soc-l2-law`.

In [ ]:
# Lab environment parameters
WORKSPACE_ID = '6cdf51ac-a467-4b2f-9d87-4bd5f69a0bec'  # customerId of soc-l2-law
LOOKBACK_HOURS = 48

from datetime import timedelta
import pandas as pd
import matplotlib.pyplot as plt
import json
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

# Configure DataFrame display options
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

credential = DefaultAzureCredential()
client = LogsQueryClient(credential)

def query_sentinel(kql_query, hours=LOOKBACK_HOURS):
    """Executes a KQL query against Log Analytics and returns a Pandas DataFrame."""
    resp = client.query_workspace(WORKSPACE_ID, kql_query, timespan=timedelta(hours=hours))
    if resp.status == LogsQueryStatus.PARTIAL:
        table = resp.partial_data[0]
    elif resp.status == LogsQueryStatus.SUCCESS:
        if not resp.tables:
            return pd.DataFrame()
        table = resp.tables[0]
    else:
        raise RuntimeError(f"Error querying Log Analytics: {resp}")
    return pd.DataFrame(data=table.rows, columns=table.columns)

print(f"[OK] LogsQueryClient initialized for Workspace: {WORKSPACE_ID}")

## 2. High-Level Summary of Ingested Telemetry

Consolidates all records carrying `SOC-L2` markers to confirm visibility across every destination table.

In [ ]:
kql_summary = '''
let win = SecurityEvent 
    | where (CommandLine has 'SOC-L2' or TargetUserName startswith 'soc-l2-p02' or TargetUserName startswith 'SOCp15' or TargetUserName startswith 'SOCg15' or MemberName has 'SOCp15')
    | summarize Total_Events = count() by Table = 'SecurityEvent (Windows/AD)';
let sys = Syslog 
    | where SyslogMessage has 'SOC-L2' or SyslogMessage has 'socp04' or SyslogMessage has 'soc_l2_p13'
    | summarize Total_Events = count() by Table = 'Syslog (Linux/Services)';
let app = AppServiceHTTPLogs 
    | where UserAgent startswith 'SOC-L2-Lab' 
    | summarize Total_Events = count() by Table = 'AppServiceHTTPLogs';
let iis = W3CIISLog 
    | where csUserAgent startswith 'SOC-L2-Lab' or csUriStem has 'soc-l2'
    | summarize Total_Events = count() by Table = 'W3CIISLog (IIS)';
let kv = AZKVAuditLogs 
    | where RequestUri has 'soc-l2' or OperationName has 'Secret'
    | summarize Total_Events = count() by Table = 'AZKVAuditLogs (Key Vault)';
let aks_audit = AKSAuditAdmin 
    | where RequestUri has 'soc-l2'
    | summarize Total_Events = count() by Table = 'AKSAuditAdmin (K8s API)';
let aks_pods = ContainerLogV2 
    | where LogMessage has 'SOC-L2'
    | summarize Total_Events = count() by Table = 'ContainerLogV2 (AKS Pods)';
let act = AzureActivity 
    | where Properties has 'SOCPrueba' or OperationNameValue has 'Microsoft.Resources/tags'
    | summarize Total_Events = count() by Table = 'AzureActivity (ARM)';
union win, sys, app, iis, kv, aks_audit, aks_pods, act
| sort by Total_Events desc
'''

df_summary = query_sentinel(kql_summary)
display(df_summary)

# Bar chart of event distribution by table
if not df_summary.empty:
    plt.figure(figsize=(10, 4))
    plt.barh(df_summary['Table'], df_summary['Total_Events'], color='#0078D4')
    plt.xlabel('Event Count')
    plt.title('Lab Security Event Distribution by Table')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 3. Windows Forensic Investigation (P02, P03, P15)

### 3.1. Test P02: Brute Force / Authentication Failures (EventID 4625)
Analyzes failed logon attempts with nonexistent marked accounts grouped by logon type.

In [ ]:
kql_p02 = '''
SecurityEvent
| where EventID == 4625 and TargetUserName startswith 'soc-l2-p02'
| extend LogonTypeDesc = case(
    LogonType == 2, 'Interactive (Console)',
    LogonType == 3, 'Network (SMB/Net)',
    LogonType == 10, 'RemoteInteractive (RDP)',
    tostring(LogonType)
  )
| summarize 
    Failed_Attempts = count(),
    First_Attempt = min(TimeGenerated),
    Last_Attempt = max(TimeGenerated)
  by Computer, TargetUserName, LogonTypeDesc, IpAddress, SubStatus
| sort by Failed_Attempts desc
'''
df_p02 = query_sentinel(kql_p02)
print(f"Distinct P02 failed authentication vectors: {len(df_p02)}")
display(df_p02)

### 3.2. Test P03: Process Execution & LOLBAS (EventID 4688)
Inspects process creation events carrying marked command-line arguments on the Windows VM.

In [ ]:
kql_p03 = r'''
SecurityEvent
| where EventID == 4688 and CommandLine has 'SOC-L2-P03'
| extend Process = tostring(split(NewProcessName, @'\')[-1])
| extend ParentProcess = tostring(split(ParentProcessName, @'\')[-1])
| project TimeGenerated, Computer, SubjectUserName, Process, ParentProcess, CommandLine
| sort by TimeGenerated desc
'''
df_p03 = query_sentinel(kql_p03)
display(df_p03)

### 3.3. Test P15: AD DS Persistence & Account Management (EventIDs 4720–4738)
Reconstructs the chronological sequence of disposable account creation, enablement, group membership changes, and deletion in Active Directory.

In [ ]:
kql_p15 = '''
SecurityEvent
| where EventID in (4720, 4722, 4724, 4725, 4726, 4727, 4728, 4729, 4730, 4732, 4733, 4738)
| where TargetUserName startswith 'SOCp15' or TargetUserName startswith 'SOCg15' or MemberName has 'SOCp15'
| extend Action = case(
    EventID == 4720, 'User Created',
    EventID == 4722, 'User Enabled',
    EventID == 4724, 'Password Reset',
    EventID == 4728, 'Added to Global Group',
    EventID == 4732, 'Added to Local Group',
    EventID == 4726, 'User Deleted',
    EventID == 4730, 'Group Deleted',
    Activity
  )
| project TimeGenerated, EventID, Action, SubjectUserName, TargetUserName, MemberName
| sort by TimeGenerated asc
'''
df_p15 = query_sentinel(kql_p15)
display(df_p15)

## 4. Linux & Web Server Investigation (P04, P04B, P11, P12, P13)

### 4.1. Linux Authentication & Privilege Events (`Syslog`)
Inspects `sudo` markers, failed `sshd` logins, and `su` attempts targeting nonexistent accounts.

In [ ]:
kql_linux = '''
Syslog
| where SyslogMessage has 'SOC-L2-P04' or (ProcessName in ('sshd', 'su', 'sudo') and SyslogMessage has 'socp04')
| project TimeGenerated, Computer, Facility, SeverityLevel, ProcessName, SyslogMessage
| sort by TimeGenerated desc
'''
df_linux = query_sentinel(kql_linux)
display(df_linux)

### 4.2. Web Server Traffic: Apache (Linux) & IIS (Windows) — P11 and P12
Compares HTTP requests recorded by Apache (`local6` in `Syslog`) and IIS (`W3CIISLog`).

In [ ]:
kql_web = '''
let apache = Syslog
    | where Facility == 'local6' and SyslogMessage has 'SOC-L2-P11'
    | project TimeGenerated, Server = 'Apache (Linux)', Method = 'HTTP', Status = '200', Detail = SyslogMessage;
let iis = W3CIISLog
    | where csUserAgent startswith 'SOC-L2-Lab' or csUriStem has 'soc-l2-p12'
    | project TimeGenerated, Server = 'IIS (Windows)', Method = csMethod, Status = tostring(scStatus), Detail = strcat(csUriStem, ' | IP: ', cIP);
union apache, iis
| sort by TimeGenerated desc
'''
df_web = query_sentinel(kql_web)
display(df_web)

## 5. Cloud Security Auditing: App Service, Key Vault & AKS (P05, P07, P08, P10)

### 5.1. Test P07: Azure Key Vault Secret Operations (`AZKVAuditLogs`)
Inspects authorized public-key secret reads (`200 OK`) and nonexistent secret probes (`404 Not Found`).

In [ ]:
kql_kv = '''
AZKVAuditLogs
| where RequestUri has 'soc-l2-p07' or OperationName has 'Secret'
| project TimeGenerated, OperationName, ResultType, ResultSignature, CallerIpAddress, RequestUri
| sort by TimeGenerated desc
'''
df_kv = query_sentinel(kql_kv)
display(df_kv)


### 5.2. Tests P08 & P10: AKS Control Plane Auditing (`AKSAuditAdmin`)

In [ ]:
kql_aks = '''
AKSAuditAdmin
| where RequestUri has 'soc-l2-p08'
| project TimeGenerated, Verb, ResponseStatus = parse_json(ResponseStatus).code, RequestUri, Username = tostring(parse_json(tostring(User)).username)
| sort by TimeGenerated desc
'''
df_aks = query_sentinel(kql_aks)
display(df_aks)

## 6. Findings Summary Matrix & CSV Export for SOC L2 Analysts

Synthesizes detection coverage across all vectors mapped to MITRE ATT&CK tactics.

In [ ]:
findings_summary = [
    {'Test': 'P02', 'Vector': 'Windows Brute Force', 'MITRE_Tactic': 'Credential Access (T1110)', 'Records': len(df_p02), 'Status': 'Detected' if len(df_p02) > 0 else 'Pending'},
    {'Test': 'P03', 'Vector': 'LOLBAS Execution', 'MITRE_Tactic': 'Execution (T1059.001)', 'Records': len(df_p03), 'Status': 'Detected' if len(df_p03) > 0 else 'Pending'},
    {'Test': 'P15', 'Vector': 'AD Account Manipulation', 'MITRE_Tactic': 'Persistence / Privilege Escalation (T1098/T1078)', 'Records': len(df_p15), 'Status': 'Detected' if len(df_p15) > 0 else 'Pending'},
    {'Test': 'P04/P04B', 'Vector': 'Linux Auth Failures', 'MITRE_Tactic': 'Initial Access / Lateral Movement (T1078)', 'Records': len(df_linux), 'Status': 'Detected' if len(df_linux) > 0 else 'Pending'},
    {'Test': 'P11/P12', 'Vector': 'Web Probes (Apache/IIS)', 'MITRE_Tactic': 'Reconnaissance (T1595)', 'Records': len(df_web), 'Status': 'Detected' if len(df_web) > 0 else 'Pending'},
    {'Test': 'P07', 'Vector': 'Key Vault Access & 404', 'MITRE_Tactic': 'Credential Access (T1555)', 'Records': len(df_kv), 'Status': 'Detected' if len(df_kv) > 0 else 'Pending'},
    {'Test': 'P08/P10', 'Vector': 'AKS API & Pod Activity', 'MITRE_Tactic': 'Execution in Container (T1610)', 'Records': len(df_aks), 'Status': 'Detected' if len(df_aks) > 0 else 'Pending'}
]

df_final_report = pd.DataFrame(findings_summary)
print("=== SOC L2 LAB SYNTHETIC EVIDENCE REPORT ===")
display(df_final_report)

# Export summary report to local CSV
df_final_report.to_csv('soc_l2_evidence_report.csv', index=False)
print("[OK] Evidence report exported to 'soc_l2_evidence_report.csv'")